# Neo4j MCP Agent with Strands Agents

Query a Neo4j graph database using **AWS Strands Agents** and **AgentCore Gateway MCP**.



## 1. Setup

In [ ]:
%pip install strands-agents strands-agents-tools mcp httpx -q

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient
from mcp.client.streamable_http import streamablehttp_client

print("Imports OK")

## 2. Configuration

Set `MODEL` and your MCP Gateway credentials.

In [ ]:
# Load configuration from CONFIG.txt
from dotenv import load_dotenv
import os

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID")
REGION = os.getenv("REGION", "us-west-2")
GATEWAY_URL = os.getenv("MCP_GATEWAY_URL")
ACCESS_TOKEN = os.getenv("MCP_ACCESS_TOKEN")

print(f"Model:   {MODEL_ID}")
print(f"Region:  {REGION}")

# Validate gateway credentials
if not GATEWAY_URL or "your-" in GATEWAY_URL:
    print("\nWARNING: Set MCP_GATEWAY_URL in CONFIG.txt before running MCP cells")
elif not ACCESS_TOKEN or "your-" in ACCESS_TOKEN:
    print("\nWARNING: Set MCP_ACCESS_TOKEN in CONFIG.txt before running MCP cells")
else:
    print(f"Gateway: {GATEWAY_URL[:50]}...")
    print("Configuration OK!")

## 3. Initialize Model & MCP Client

**Key pattern from AWS sample:** The transport factory returns a fresh `streamablehttp_client` each time, with the Bearer token embedded in headers.

In [ ]:
# Bedrock model
model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0,
)


# Token getter (called each time transport is created)
def get_token():
    return ACCESS_TOKEN


# Transport factory - returns fresh streamablehttp_client each call
def create_streamable_http_transport():
    return streamablehttp_client(
        GATEWAY_URL,
        headers={"Authorization": f"Bearer {get_token()}"}
    )


# MCP client with transport factory
mcp_client = MCPClient(create_streamable_http_transport)

print(f"Model initialized: {MODEL_ID}")
print("MCP client ready")

## 4. Test MCP Connection

In [ ]:
with mcp_client:
    tools = mcp_client.list_tools_sync()
    print(f"Connected! Found {len(tools)} tools:")
    for tool in tools:
        print(f"  - {tool.tool_spec['name']}")

## 5. Create Agent & Query Function

In [ ]:
SYSTEM_PROMPT = """You are a Neo4j database assistant. You can:
- Get the database schema
- Run read-only Cypher queries

Always get the schema first, then query based on actual labels/relationships.
Be concise. Format results clearly."""


def query(question: str) -> str:
    """Query the Neo4j database via MCP."""
    print(f"Q: {question}")
    print("-" * 60)
    
    with mcp_client:
        tools = mcp_client.list_tools_sync()
        agent = Agent(
            model=model,
            tools=tools,
            system_prompt=SYSTEM_PROMPT,
        )
        result = agent(question)
    
    print(f"\nA: {result}")
    return str(result)

## 6. Demo Queries

In [ ]:
_ = query("What is the database schema?")

In [ ]:
_ = query("How many nodes are there by label?")

In [ ]:
_ = query("Show 5 sample records from the most populated node type.")

## 7. Your Query

In [ ]:
# _ = query("Your question here")